# FitNote Trainer 제출용 DB 분석 노트북

이 노트북은 `fitnote_trainer_submission.db` SQLite DB를 기준으로 샘플 데이터 확인과 분석 쿼리를 실행합니다.

## 1. DB 연결 및 기본 함수

In [ ]:

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = Path('fitnote_trainer_submission.db')
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

def q(sql, title=None):
    if title:
        print(f'\n## {title}')
    return pd.read_sql_query(sql, conn)


## 2. 제출 DB 테이블 확인

In [ ]:

q('''
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
''', 'DB 테이블 목록')


## 3. 분석 쿼리 6개

요구사항인 분석 쿼리 5개 이상을 충족하기 위해 6개 쿼리를 포함했습니다.

In [ ]:

# 1. 회원별 최근 수업/자세 점수/진행 중 과제 요약
q1 = q('''
SELECT
  m.id AS member_id,
  m.name AS member_name,
  m.fitness_goal,
  MAX(s.session_date) AS last_session_date,
  ROUND(AVG(ps.total_score), 1) AS avg_posture_score,
  SUM(CASE WHEN a.status IN ('unchecked', 'in_progress') THEN 1 ELSE 0 END) AS open_assignment_count
FROM members m
LEFT JOIN sessions s ON s.member_id = m.id
LEFT JOIN posture_images pi ON pi.member_id = m.id
LEFT JOIN posture_analysis pa ON pa.posture_image_id = pi.id
LEFT JOIN posture_scores ps ON ps.analysis_id = pa.id
LEFT JOIN assignments a ON a.member_id = m.id
GROUP BY m.id, m.name, m.fitness_goal
ORDER BY avg_posture_score ASC;
''', '1. 회원별 요약 대시보드')
display(q1)

# 2. 자세 점수 낮은 회원과 개선 포인트
q2 = q('''
SELECT
  m.name AS member_name,
  pa.movement_name,
  ps.total_score,
  pa.main_issue,
  ps.improvement_points,
  pa.analyzed_at
FROM posture_scores ps
JOIN posture_analysis pa ON pa.id = ps.analysis_id
JOIN posture_images pi ON pi.id = pa.posture_image_id
JOIN members m ON m.id = pi.member_id
WHERE ps.total_score < 75
ORDER BY ps.total_score ASC;
''', '2. 자세 점수 75점 미만 분석')
display(q2)

# 3. 운동 종목별 수행 볼륨 요약
q3 = q('''
SELECT
  se.exercise_name,
  COUNT(*) AS record_count,
  SUM(COALESCE(se.sets, 0) * COALESCE(se.reps, 0)) AS total_reps,
  ROUND(AVG(se.weight_kg), 1) AS avg_weight_kg
FROM session_exercises se
GROUP BY se.exercise_name
ORDER BY total_reps DESC;
''', '3. 운동 종목별 수행 볼륨')
display(q3)

# 4. 과제 완료율
q4 = q('''
SELECT
  m.name AS member_name,
  COUNT(a.id) AS assignment_count,
  SUM(CASE WHEN a.status = 'completed' THEN 1 ELSE 0 END) AS completed_count,
  ROUND(100.0 * SUM(CASE WHEN a.status = 'completed' THEN 1 ELSE 0 END) / COUNT(a.id), 1) AS completion_rate
FROM members m
JOIN assignments a ON a.member_id = m.id
GROUP BY m.id, m.name
ORDER BY completion_rate DESC;
''', '4. 회원별 과제 완료율')
display(q4)

# 5. 트레이너별 수업 수와 담당 회원 수
q5 = q('''
SELECT
  t.name AS trainer_name,
  COUNT(DISTINCT m.id) AS member_count,
  COUNT(DISTINCT s.id) AS session_count,
  COUNT(DISTINCT r.id) AS report_count
FROM trainers t
LEFT JOIN members m ON m.trainer_id = t.id
LEFT JOIN sessions s ON s.trainer_id = t.id
LEFT JOIN reports r ON r.member_id = m.id
GROUP BY t.id, t.name
ORDER BY session_count DESC;
''', '5. 트레이너별 사용량')
display(q5)

# 6. 날짜별 수업 일정
q6 = q('''
SELECT
  s.session_date,
  t.name AS trainer_name,
  COUNT(*) AS session_count,
  GROUP_CONCAT(m.name || '(' || s.focus_area || ')', ', ') AS session_members
FROM sessions s
JOIN members m ON m.id = s.member_id
JOIN trainers t ON t.id = s.trainer_id
GROUP BY s.session_date, t.name
ORDER BY s.session_date;
''', '6. 날짜별 수업 일정')
display(q6)


## 4. 시각화 예시

Colab에서 한글이 깨지면 런타임에 한글 폰트를 설치한 뒤 런타임을 재시작하고 다시 실행하면 됩니다.

In [ ]:

# 그래프 1. 회원별 평균 자세 점수
score_df = q('''
SELECT m.name AS member_name, ROUND(AVG(ps.total_score), 1) AS avg_score
FROM members m
JOIN posture_images pi ON pi.member_id = m.id
JOIN posture_analysis pa ON pa.posture_image_id = pi.id
JOIN posture_scores ps ON ps.analysis_id = pa.id
GROUP BY m.id, m.name
ORDER BY avg_score DESC;
''')

plt.figure(figsize=(8, 4))
plt.bar(score_df['member_name'], score_df['avg_score'])
plt.ylim(0, 100)
plt.title('회원별 평균 자세 점수')
plt.xlabel('회원')
plt.ylabel('평균 점수')
plt.show()

# 그래프 2. 박수진 회원의 자세 점수 변화
trend_df = q('''
SELECT pa.analyzed_at, ps.total_score
FROM posture_scores ps
JOIN posture_analysis pa ON pa.id = ps.analysis_id
JOIN posture_images pi ON pi.id = pa.posture_image_id
JOIN members m ON m.id = pi.member_id
WHERE m.name = '박수진'
ORDER BY pa.analyzed_at;
''')

plt.figure(figsize=(8, 4))
plt.plot(trend_df['analyzed_at'], trend_df['total_score'], marker='o')
plt.ylim(0, 100)
plt.title('박수진 회원 자세 점수 변화')
plt.xlabel('분석일시')
plt.ylabel('점수')
plt.xticks(rotation=25)
plt.show()


## 5. 종료

In [ ]:
conn.close()
print('분석 완료')
